In [14]:
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

In [16]:
model_checkpoint = "vinai/bertweet-base"

tokenizer = AutoTokenizer.from_pretrained(
    model_checkpoint,
    use_fast=True,
    normalization=True,
    add_prefix_space=True
)

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


In [17]:
def read_iob2(filepath, masked_test=False):
    sentences = []
    labels = []

    current_tokens = []
    current_labels = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                if current_tokens:
                    sentences.append(current_tokens)
                    labels.append(current_labels)
                    current_tokens = []
                    current_labels = []
                continue

            if line.startswith("#"):
                continue

            parts = line.split()

            token = parts[1]

            if masked_test:
                tag = "O"
            else:
                tag = parts[2]

            current_tokens.append(token)
            current_labels.append(tag)

    if current_tokens:
        sentences.append(current_tokens)
        labels.append(current_labels)

    return {"tokens": sentences, "ner_tags": labels}

In [18]:
train_data = read_iob2("en_ewt-ud-train.iob2")
dev_data = read_iob2("en_ewt-ud-dev.iob2")

dataset = DatasetDict({
    "train": Dataset.from_dict(train_data),
    "validation": Dataset.from_dict(dev_data),
})

dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 12543
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2001
    })
})

In [19]:
all_labels = set()

for split in ["train", "validation"]:
    for sentence_labels in dataset[split]["ner_tags"]:
        all_labels.update(sentence_labels)

label_list = sorted(all_labels)

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(label_list)
print(label2id)

['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']
{'B-LOC': 0, 'B-ORG': 1, 'B-PER': 2, 'I-LOC': 3, 'I-ORG': 4, 'I-PER': 5, 'O': 6}


In [20]:
def encode_labels(example):
    example["labels"] = [label2id[label] for label in example["ner_tags"]]
    return example

dataset = dataset.map(encode_labels)

Map:   0%|          | 0/12543 [00:00<?, ? examples/s]

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

In [23]:
def tokenize_and_align_labels(example):
    tokens = []
    labels = []

    for word, label in zip(example["tokens"], example["labels"]):
        word_tokens = tokenizer.tokenize(word)

        if len(word_tokens) == 0:
            continue

        tokens.extend(word_tokens)

        labels.append(label)

        labels.extend([-100] * (len(word_tokens) - 1))

    max_length = 128
    tokens = tokens[:max_length - 2]
    labels = labels[:max_length - 2]

    input_ids = tokenizer.convert_tokens_to_ids(tokens)

    input_ids = [tokenizer.bos_token_id] + input_ids + [tokenizer.eos_token_id]
    labels = [-100] + labels + [-100]

    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


tokenized_dataset = dataset.map(tokenize_and_align_labels)

Map:   0%|          | 0/12543 [00:00<?, ? examples/s]

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

In [24]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
classifier.bias                 | MISSING    | 
classifier.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

In [25]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [27]:
training_args = TrainingArguments(
    output_dir="./bertweet_ewt_model",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100
)

In [28]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):
        current_predictions = []
        current_labels = []

        for p, l in zip(prediction, label):
            if l != -100:
                current_predictions.append(id2label[p])
                current_labels.append(id2label[l])

        true_predictions.append(current_predictions)
        true_labels.append(current_labels)

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

In [30]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [31]:
trainer.train()

c:\Users\LG\anaconda3\envs\ner\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
100,0.355313
200,0.133732
300,0.102220
400,0.078362
500,0.063977
600,0.069567
700,0.067388
800,0.058995
900,0.051367
1000,0.049294


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4704, training_loss=0.03914754705972412, metrics={'train_runtime': 7305.5626, 'train_samples_per_second': 5.151, 'train_steps_per_second': 0.644, 'total_flos': 800375896709136.0, 'train_loss': 0.03914754705972412, 'epoch': 3.0})

In [33]:
predictions_output = trainer.predict(tokenized_dataset["validation"])

predictions = np.argmax(predictions_output.predictions, axis=2)
labels = predictions_output.label_ids

true_predictions = []
true_labels = []

for prediction, label in zip(predictions, labels):
    current_predictions = []
    current_labels = []

    for p, l in zip(prediction, label):
        if l != -100:
            current_predictions.append(id2label[p])
            current_labels.append(id2label[l])

    true_predictions.append(current_predictions)
    true_labels.append(current_labels)

print("Precision:", precision_score(true_labels, true_predictions))
print("Recall:", recall_score(true_labels, true_predictions))
print("F1:", f1_score(true_labels, true_predictions))
print(classification_report(true_labels, true_predictions))

Precision: 0.8215725806451613
Recall: 0.8436853002070394
F1: 0.8324821246169561
              precision    recall  f1-score   support

         LOC       0.84      0.84      0.84       399
         ORG       0.69      0.72      0.71       224
         PER       0.89      0.92      0.91       343

   micro avg       0.82      0.84      0.83       966
   macro avg       0.81      0.83      0.82       966
weighted avg       0.82      0.84      0.83       966



In [34]:
def read_tweebank_bio(path):
    sentences = []
    labels = []

    current_tokens = []
    current_labels = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line == "":
                if current_tokens:
                    sentences.append(current_tokens)
                    labels.append(current_labels)
                    current_tokens = []
                    current_labels = []
                continue

            if line.startswith("-DOCSTART-"):
                continue

            parts = line.split()

            token = parts[0]
            tag = parts[-1]

            # Same as cross-domain notebook: remove MISC because our model only has LOC/ORG/PER/O
            if tag in ["B-MISC", "I-MISC"]:
                tag = "O"

            current_tokens.append(token)
            current_labels.append(tag)

    if current_tokens:
        sentences.append(current_tokens)
        labels.append(current_labels)

    return sentences, labels

In [35]:
tweebank_tokens, tweebank_labels = read_tweebank_bio("Tweebank test.bio")

print("Tweebank sentences:", len(tweebank_tokens))
print(sorted(set(label for sent in tweebank_labels for label in sent)))
print(tweebank_tokens[0])
print(tweebank_labels[0])

Tweebank sentences: 1201
['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']
['@USER1812', 'No', ',', 'I', "'m", 'not', '.', 'It', "'s", 'definitely', 'not', 'a', 'rapper', '.']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [36]:
tweebank_dataset = Dataset.from_dict({
    "tokens": tweebank_tokens,
    "ner_tags": tweebank_labels
})

def encode_tweebank_labels(example):
    example["labels"] = [
        label2id[label] if label in label2id else label2id["O"]
        for label in example["ner_tags"]
    ]
    return example

tweebank_dataset = tweebank_dataset.map(encode_tweebank_labels)

Map:   0%|          | 0/1201 [00:00<?, ? examples/s]

In [37]:
tokenized_tweebank = tweebank_dataset.map(tokenize_and_align_labels)

Map:   0%|          | 0/1201 [00:00<?, ? examples/s]

In [38]:
tweebank_predictions = trainer.predict(tokenized_tweebank)

predictions = np.argmax(tweebank_predictions.predictions, axis=2)
labels = tweebank_predictions.label_ids

true_predictions = []
true_labels = []

for prediction, label in zip(predictions, labels):
    current_predictions = []
    current_labels = []

    for p, l in zip(prediction, label):
        if l != -100:
            current_predictions.append(id2label[p])
            current_labels.append(id2label[l])

    true_predictions.append(current_predictions)
    true_labels.append(current_labels)

print("Precision:", precision_score(true_labels, true_predictions))
print("Recall:", recall_score(true_labels, true_predictions))
print("F1:", f1_score(true_labels, true_predictions))
print(classification_report(true_labels, true_predictions))

Precision: 0.7097902097902098
Recall: 0.7173144876325088
F1: 0.7135325131810193
              precision    recall  f1-score   support

         LOC       0.58      0.85      0.69       111
         ORG       0.66      0.41      0.51       182
         PER       0.80      0.87      0.83       273

   micro avg       0.71      0.72      0.71       566
   macro avg       0.68      0.71      0.68       566
weighted avg       0.71      0.72      0.70       566

